# Train CausaGanha Decision Segmenter (OPF)

Fine-tune OpenAI Privacy Filter as a 25-class anchor-span segmenter for
Brazilian judicial decisions.

**Architecture:** This is the TRAINING notebook. It consumes a frozen artifact
set (`train.jsonl`, `val.jsonl`, `test.jsonl`, `label_space.json`) and never
re-derives inputs. Data prep is done separately (CPU script or prep notebook).

**Ontology (v7):** 6 single-anchor + 9 start/end pairs (18) + O = 25 entries.
Short anchor spans for tiling regions, `_inicio`/`_fim` pairs for discrete regions.

**Requirements:** GPU runtime (T4 or better). Go to Runtime → Change runtime type → T4 GPU.

## 1. Mount Drive + set paths

Drive is the persistence layer. Base model stored once, each run's checkpoint saved immediately.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# Drive layout (per colab-and-drive.md)
ROOT = "/content/drive/MyDrive/opf-finetune"
CATEGORY_VERSION = "segmenter_v7"

DRIVE_BASE = f"{ROOT}/base/privacy_filter"
DRIVE_DATA = f"{ROOT}/data/{CATEGORY_VERSION}"
DRIVE_CKPT_ROOT = f"{ROOT}/checkpoints/{CATEGORY_VERSION}"

# Local working paths
LOCAL_BASE = "/content/base/privacy_filter"
LOCAL_DATA = f"/content/data/{CATEGORY_VERSION}"
LOCAL_OUT = "/content/out/best"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Drive root: {ROOT}")
print(f"Category version: {CATEGORY_VERSION}")

## 2. Install OPF

In [ ]:
!uv pip install --system "opf @ git+https://github.com/openai/privacy-filter.git" httpx

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Switch to GPU runtime!"

## 3. Restore-or-fetch base model

Download once (~2.8 GB), then persist to Drive for all future runs.

In [ ]:
import shutil

if os.path.exists(f"{LOCAL_BASE}/config.json"):
    print("Base model already local this session")
elif os.path.exists(f"{DRIVE_BASE}/config.json"):
    print("Restoring base model from Drive...")
    shutil.copytree(DRIVE_BASE, LOCAL_BASE)
    print("Restored.")
else:
    print("First time: downloading base model...")
    from opf._common.checkpoint_download import ensure_default_checkpoint
    ensure_default_checkpoint()
    # Find where OPF saved it and copy to our paths
    from opf._common.constants import DEFAULT_MODEL_PATH
    default_path = str(DEFAULT_MODEL_PATH)
    if os.path.exists(default_path):
        shutil.copytree(default_path, LOCAL_BASE, dirs_exist_ok=True)
    # Persist to Drive
    os.makedirs(os.path.dirname(DRIVE_BASE), exist_ok=True)
    shutil.copytree(LOCAL_BASE, DRIVE_BASE)
    print(f"Base model saved to Drive: {DRIVE_BASE}")

## 4. Load frozen data artifacts

Artifacts come from Drive (uploaded by prep script/CI) or from the repo.

In [ ]:
import json

# Check Drive first, then fall back to downloading from repo
REQUIRED = ["train.jsonl", "val.jsonl", "test.jsonl", "label_space.json"]

if all(os.path.exists(f"{DRIVE_DATA}/{f}") for f in REQUIRED):
    print(f"Using cached data from Drive: {DRIVE_DATA}")
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for f in REQUIRED + ["manifest.json", "opf_annotate.py"]:
        src = f"{DRIVE_DATA}/{f}"
        dst = f"{LOCAL_DATA}/{f}"
        if os.path.exists(src):
            shutil.copy2(src, dst)
else:
    print("No cached data on Drive. Cloning repo to generate...")
    !git clone --depth 1 https://github.com/franklinbaldo/causaganha.git /content/causaganha
    !uv pip install --system ibis-framework[duckdb] structlog pyarrow
    os.environ["PYTHONPATH"] = "/content/causaganha"

    import sys
    sys.path.insert(0, "/content/causaganha")
    import subprocess
    subprocess.run([
        sys.executable, "/content/causaganha/scripts/prepare_privacy_filter_dataset.py",
        "--output-dir", LOCAL_DATA,
    ], check=True)

    # Cache data + validator to Drive
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in os.listdir(LOCAL_DATA):
        shutil.copy2(f"{LOCAL_DATA}/{f}", f"{DRIVE_DATA}/{f}")
    # Cache the validator script alongside data
    validator_src = "/content/causaganha/scripts/opf_annotate.py"
    if os.path.exists(validator_src):
        shutil.copy2(validator_src, f"{DRIVE_DATA}/opf_annotate.py")
        shutil.copy2(validator_src, f"{LOCAL_DATA}/opf_annotate.py")
    print(f"Data cached to Drive: {DRIVE_DATA}")

# Show manifest
manifest_path = f"{LOCAL_DATA}/manifest.json"
if os.path.exists(manifest_path):
    manifest = json.load(open(manifest_path))
    print(f"\nManifest: {json.dumps(manifest, indent=2)}")

# Load and verify label space
ls = json.load(open(f"{LOCAL_DATA}/label_space.json"))
assert ls["span_class_names"][0] == "O", "O must be first in span_class_names"
print(f"\nLabel space: {ls['span_class_names']}")
print(f"Categories: {len(ls['span_class_names'])} ({len(ls['span_class_names'])-1} + O)")

## 5. Validate splits

In [ ]:
import sys, subprocess

# Locate the validator: repo clone > local data dir > download from GitHub
validator = "/content/causaganha/scripts/opf_annotate.py"
if not os.path.exists(validator):
    validator = f"{LOCAL_DATA}/opf_annotate.py"
if not os.path.exists(validator):
    # Download from repo (handles Drive-cached data without repo clone)
    import urllib.request
    validator = f"{LOCAL_DATA}/opf_annotate.py"
    url = "https://raw.githubusercontent.com/franklinbaldo/causaganha/main/scripts/opf_annotate.py"
    print(f"Downloading validator from {url}")
    urllib.request.urlretrieve(url, validator)
    # Cache to Drive for future runs
    shutil.copy2(validator, f"{DRIVE_DATA}/opf_annotate.py")

for split in ["train", "val", "test"]:
    jsonl = f"{LOCAL_DATA}/{split}.jsonl"
    print(f"\n--- Validating {split} ---")
    result = subprocess.run([
        sys.executable, validator, "validate", jsonl,
        "--label-space", f"{LOCAL_DATA}/label_space.json",
    ])
    assert result.returncode == 0, f"Validation failed for {split}!"
print("\nAll splits valid.")

## 6. Train with OPF

No hardcoded `--n-ctx` — let OPF use its full 128k context window.
On T4 (16GB VRAM), batch_size=1 is safe. On A100 (40GB), try batch_size=4.

In [ ]:
import sys, subprocess, time

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
EPOCHS = 3
BATCH_SIZE = 1  # T4-safe; increase on A100

os.makedirs(LOCAL_OUT, exist_ok=True)

cmd = [
    sys.executable, "-m", "opf", "train",
    f"{LOCAL_DATA}/train.jsonl",
    "--validation-dataset", f"{LOCAL_DATA}/val.jsonl",
    "--label-space-json", f"{LOCAL_DATA}/label_space.json",
    "--checkpoint", LOCAL_BASE,
    "--output-dir", LOCAL_OUT,
    "--device", "cuda",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
]
print(f"Training command: {' '.join(cmd)}")
print(f"Run ID: {RUN_ID}")

result = subprocess.run(cmd)
assert result.returncode == 0, f"Training failed with rc={result.returncode}"
print(f"\nTraining complete. Checkpoint: {LOCAL_OUT}")

## 7. Save checkpoint to Drive immediately

Colab runtimes are ephemeral — save before anything else.

In [ ]:
import shutil

DRIVE_RUN = f"{DRIVE_CKPT_ROOT}/{RUN_ID}"
os.makedirs(DRIVE_RUN, exist_ok=True)
shutil.copytree(LOCAL_OUT, DRIVE_RUN, dirs_exist_ok=True)
print(f"Checkpoint saved to Drive: {DRIVE_RUN}")

## 8. Evaluate on test set

In [ ]:
import sys, subprocess, json

METRICS_PATH = f"{LOCAL_OUT}/test_metrics.json"

cmd = [
    sys.executable, "-m", "opf", "eval",
    f"{LOCAL_DATA}/test.jsonl",
    "--checkpoint", LOCAL_OUT,
    "--device", "cuda",
    "--per-class",
    "--metrics-out", METRICS_PATH,
]
result = subprocess.run(cmd)

if result.returncode == 0 and os.path.exists(METRICS_PATH):
    metrics = json.load(open(METRICS_PATH))

    # OPF uses flat keys like "detection.span.f1" and "by_class.<label>.span.f1";
    # fall back to sklearn-style "macro avg" / per-category dicts.
    macro_f1 = metrics.get("detection.span.f1") or metrics.get("macro avg", {}).get("f1-score", 0)
    print(f"\nMacro F1: {macro_f1:.3f}")
    print()

    for cat in ls["span_class_names"]:
        if cat == "O":
            continue
        f1 = metrics.get(f"by_class.{cat}.span.f1")
        p = metrics.get(f"by_class.{cat}.span.precision")
        r = metrics.get(f"by_class.{cat}.span.recall")
        n = metrics.get(f"by_class.{cat}.support")
        if f1 is not None:
            print(f"  {cat:<22} P={p or 0:.2f}  R={r or 0:.2f}  F1={f1:.2f}  n={n or 0}")
        else:
            m = metrics.get(cat, {})
            if m:
                print(f"  {cat:<22} P={m.get('precision',0):.2f}  R={m.get('recall',0):.2f}  F1={m.get('f1-score',0):.2f}  n={m.get('support',0)}")

    # Save metrics alongside checkpoint on Drive
    shutil.copy2(METRICS_PATH, f"{DRIVE_RUN}/test_metrics.json")
    print(f"\nMetrics saved to: {DRIVE_RUN}/test_metrics.json")
else:
    print(f"Evaluation failed (rc={result.returncode})")

## 9. Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Category version: {CATEGORY_VERSION}")
print(f"Run ID: {RUN_ID}")
print(f"Checkpoint (Drive): {DRIVE_RUN}")
print(f"Data (Drive): {DRIVE_DATA}")
print(f"\nDrive tree:")
print(f"  {ROOT}/")
print(f"  ├── base/privacy_filter/")
print(f"  ├── data/{CATEGORY_VERSION}/")
print(f"  └── checkpoints/{CATEGORY_VERSION}/{RUN_ID}/")